# TP 3 — Partie 4 : exposer le Lakehouse avec Trino et Superset

**Objectif :** donner aux équipes métier un accès **en SQL** et un **tableau de bord d'entreprise**, sans passer par Spark ni par un notebook.

- **Trino** est un moteur SQL distribué, fait pour les requêtes interactives. Il lit les **mêmes tables Iceberg** que Spark, sans les copier : le stockage et le calcul sont séparés. Il sait aussi interroger d'autres sources, comme la base PostgreSQL du TP 1, dans une même requête (*fédération*).
- **Superset** est un outil de BI open source, l'équivalent de Power BI ou Tableau : il se connecte à Trino et permet de construire des tableaux de bord sans écrire de code.

> **⚠️ Mémoire.** Arrêtez Airflow et les noyaux des autres notebooks, puis démarrez Trino et Superset depuis le terminal **Ubuntu** :
> ```bash
> docker compose --profile orchestration stop airflow
> docker compose --profile full up -d
> ```
> Ce notebook n'utilise pas Spark : seulement Trino.

In [ ]:
import pandas as pd
import trino

cnx = trino.dbapi.connect(host="trino", port=8080, user="etudiant", catalog="lakehouse")


def sql(requete):
    """Exécute une requête Trino et renvoie un DataFrame pandas."""
    cur = cnx.cursor()
    cur.execute(requete)
    return pd.DataFrame(cur.fetchall(), columns=[c[0] for c in cur.description])


def a_completer(quoi):
    """Signale clairement une partie du TP qui n'a pas encore été écrite."""
    raise NotImplementedError(f"⚠️ Cellule à compléter : {quoi}")


sql("SHOW CATALOGS")

## Étape 1 — Les mêmes tables, un autre moteur

Les tables construites par Spark et dbt sont visibles telles quelles. Notez la syntaxe : `catalogue.schéma.table`.

In [ ]:
print(sql("SHOW TABLES FROM lakehouse.gold").to_string())
sql("""
    SELECT d.annee_mois, COUNT(*) AS nb_avis, ROUND(AVG(a.note), 2) AS note_moyenne
    FROM lakehouse.gold.fait_avis a
    JOIN lakehouse.gold.dim_date d ON d.date_id = a.date_avis_id
    GROUP BY d.annee_mois ORDER BY d.annee_mois DESC LIMIT 6
""")

## Étape 2 — La fédération : PostgreSQL et Iceberg dans une même requête

Le catalogue `postgres` donne accès, en direct, à la base de l'application de vente du TP 1 (schéma `ventes`). Écrivez une requête qui compare, par statut de commande, le nombre de commandes **dans la base source** (`postgres.ventes.orders`) et **dans le Lakehouse** (`lakehouse.silver.orders`).

In [ ]:
a_completer("requête fédérée PostgreSQL / Iceberg")  # À COMPLÉTER

> **❓ Question 1.** Expliquez les écarts entre la source et le Lakehouse. La fédération permet de joindre des sources sans les copier : pourquoi ne remplace-t-elle pas pour autant l'ingestion dans le Lakehouse ?

## Étape 3 — Le temps et les métadonnées, en SQL

Trino lit aussi les métadonnées Iceberg. Les versions d'une table sont dans la table spéciale `"nom$snapshots"`, et `FOR VERSION AS OF` relit une version passée.

In [ ]:
versions = sql('SELECT committed_at, snapshot_id, operation FROM lakehouse.silver."orders$snapshots" ORDER BY committed_at')
print(versions.to_string())
premiere = versions.snapshot_id.iloc[0]
sql(f"""SELECT 'aujourd''hui' AS version, COUNT(*) AS commandes FROM lakehouse.silver.orders
        UNION ALL
        SELECT 'lot 1', COUNT(*) FROM lakehouse.silver.orders FOR VERSION AS OF {premiere}""")

## Étape 4 — Le catalogue photo vu par la direction

Écrivez la requête qui servira au tableau de bord : pour chaque vendeur, le nombre de photos reçues, la part de photos rejetées et le nombre de fiches signalées comme incohérentes (`gold.photos_controle`).

In [ ]:
a_completer("indicateurs du catalogue par vendeur")  # À COMPLÉTER

## Étape 5 — Le tableau de bord Superset

Ouvrez http://localhost:8088 (identifiant `admin`, mot de passe `admin`) et suivez les étapes du support (section « Le tableau de bord ») :
1. **Connecter la base** : *Settings > Database Connections > + Database*, type *Trino*, adresse `trino://superset@trino:8080/lakehouse`.
2. **Déclarer les jeux de données** (*Datasets*) : `gold.fait_avis`, `gold.fait_livraisons`, `gold.photos_controle`, et une requête de **SQL Lab** enregistrée comme jeu de données (la requête de l'étape 4).
3. **Construire les graphiques**, puis les assembler dans un tableau de bord avec un **filtre** commun.

> **❓ Question 2.** Comparez Superset au tableau de bord Streamlit des TP 1 et 2 : qui peut le modifier, comment le partager, et quand préférer l'un ou l'autre ?